# Project 02: Single-Turn RLVR via Group Relative Policy Optimization (GRPO)

This project implements an end-to-end post-training pipeline aligning a small language model (`Qwen/Qwen2.5-0.5B-Instruct`) to reason step-by-step using **Reinforcement Learning with Verifiable Rewards (RLVR)**.

---

### System Architecture & Pipeline Overview

```
┌───────────────────────────────┐
│   Prompt Dataset (dataset.py) │
│   • Multi-step math problems  │
│   • System reasoning prompt   │
└──────────────┬────────────────┘
               │
               ▼
┌───────────────────────────────┐
│   Base Policy (Qwen2.5 + LoRA)│
│   Samples G=4 rollouts/prompt │
└──────────────┬────────────────┘
               │
   ┌───────────┼───────────┐
   ▼           ▼           ▼
┌─────────────────┐ ┌─────────────────┐ ┌─────────────────┐
│ Format Reward   │ │ Accuracy Reward │ │ Length Sanity   │
│ (w = 0.3)       │ │ (w = 1.0)       │ │ (w = 0.1)       │
│ Checks <think>  │ │ SymPy numeric   │ │ Minimum step    │
│ tags            │ │ match           │ │ depth           │
└────────┬────────┘ └────────┬────────┘ └────────┬────────┘
         │                   │                   │
         └───────────┬───────┴───────────────────┘
                     │
                     ▼
         ┌───────────────────────────────┐
         │  GRPOTrainer (TRL Engine)     │
         │  • Normalizes group advantage │
         │  • Enforces KL penalty (β)    │
         │  • Backpropagates LoRA grads  │
         └──────────────┬────────────────┘
                        │
                        ▼
         ┌───────────────────────────────┐
         │  Evaluation (evaluate.py)     │
         │  • Measures Pass@1 & Pass@4   │
         │  • Tracks reasoning length    │
         └───────────────────────────────┘
```

┌───────────────────────────────┐

### Module & File Breakdown

#### 1. `dataset.py` — Prompt Curation & Reasoning Template Contracts
* **Purpose:** Generates synthetic, deterministic arithmetic reasoning problems (e.g., `(a * 2) + b`, `(b * c) - a`) with known integer ground truths.
* **Prompt Contract:** Injects a standard system prompt instructing the model to generate its intermediate chain of thought inside `<think>...</think>` tags and its final result strictly inside `<answer>...</answer>` tags.
* **Output:** Returns Hugging Face `Dataset` objects split into training and evaluation sets, structured with standard conversational message lists.

---

#### 2. `verifiers.py` — Deterministic Verifiable Reward Functions (RLVR)
Eliminates neural reward model ambiguity by calculating rewards purely through deterministic rule engines:

* **`format_reward_func` ($w = 0.3$):**
  Uses regular expressions (`r"^<think>.*?</think>\s*<answer>.*?</answer>$"`) to verify that the model cleanly separated its thinking process from its answer without malformed or missing tags.
* **`accuracy_reward_func` ($w = 1.0$):**
  Extracts the content of `<answer>`, normalizes whitespace/punctuation, and runs **SymPy symbolic equality** (`sympy.sympify(pred) == sympy.sympify(truth)`). This ensures exact mathematical equivalence without relying on brittle string matching.
* **`length_sanity_reward_func` ($w = 0.1$):**
  Inspects the character length inside `<think>` to ensure the model actually performed non-trivial reasoning steps rather than emitting empty tags.

---

#### 3. `evaluate.py` — Benchmark & Metric Tracking
* **Purpose:** Evaluates the model on the held-out test split before and after GRPO fine-tuning.
* **Key Metrics:**
  * **Pass@1:** Accuracy of a single sampled solution.
  * **Pass@k (e.g., Pass@4):** Probability that at least one of $k$ sampled stochastic reasoning paths reaches the verified correct answer.
  * **Average Reasoning Length:** Measures word count within `<think>` tags to track whether the policy develops explicit search/reasoning behavior during RL.

---

#### 4. Training Engine & Notebook Orchestration (`GRPOTrainer` + `PEFT`)
* **Parameter-Efficient Fine-Tuning (LoRA):** Injects low-rank adapters ($r=16, \alpha=32$) into all attention and MLP projection heads (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`), keeping the base model frozen to save VRAM and prevent catastrophic forgetting.
* **Group Relative Normalization:** Samples $G=4$ parallel rollouts per prompt, computes $A_i = \frac{r_i - \mu}{\sigma + \epsilon}$, and updates policy parameters without requiring a Critic network.
* **KL Divergence Monitoring:** Constrains policy drift via penalty coefficient $\beta =

In [1]:
import sys

from dataset import generate_math_dataset
from verifiers import format_reward_func, accuracy_reward_func, length_sanity_reward_func

train_ds, test_ds = generate_math_dataset(num_samples=100)
print(f"Dataset generated: {len(train_ds)} train examples, {len(test_ds)} test examples.")

# Test verifier
mock_completion = [[{"content": "<think>We compute (20 * 2) + 10 = 40 + 10 = 50.</think>\n<answer>50</answer>"}]]
mock_gt = ["50"]
print("Format Score:  ", format_reward_func(mock_completion)[0])
print("Accuracy Score:", accuracy_reward_func(mock_completion, ground_truth=mock_gt)[0])
print("Length Score:  ", length_sanity_reward_func(mock_completion)[0])

c:\WHATEVERELSE\GeekStuff\conda\envs\pygpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset generated: 80 train examples, 20 test examples.
Format Score:   0.3
Accuracy Score: 1.0
Length Score:   0.1


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# 1. Load Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

# 2. Configure LoRA Adapter
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 3. Configure GRPO Hyperparameters
training_args = GRPOConfig(
    output_dir="./02_grpo_single_turn_rlvr/output",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_generations=4,          # Group size G = 4 parallel rollouts per prompt
    max_prompt_length=128,
    max_completion_length=200,
    temperature=0.8,
    beta=0.04,                  # KL penalty coefficient
    max_steps=40,               # Set to 30-50 steps for a fast demo run
    logging_steps=5,
    save_steps=20,
    report_to="none"
)

# 4. Telemetry Callback to record metrics for notebook plotting
telemetry_history = {"step": [], "reward_mean": [], "kl": []}

class TelemetryCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            if "reward" in logs:
                telemetry_history["reward_mean"].append(logs["reward"])
            elif "rewards/mean" in logs:
                telemetry_history["reward_mean"].append(logs["rewards/mean"])
                
            if "kl" in logs:
                telemetry_history["kl"].append(logs["kl"])
            elif "objective/kl" in logs:
                telemetry_history["kl"].append(logs["objective/kl"])
                
            if "step" in logs:
                telemetry_history["step"].append(logs["step"])
            elif state.global_step not in telemetry_history["step"]:
                telemetry_history["step"].append(state.global_step)

# 5. Initialize GRPOTrainer
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[format_reward_func, accuracy_reward_func, length_sanity_reward_func],
    args=training_args,
    train_dataset=train_ds,
    peft_config=peft_config,
    callbacks=[TelemetryCallback()]
)

print("Starting GRPO Training...")
trainer.train()
print("Training Complete!")

c:\WHATEVERELSE\GeekStuff\conda\envs\pygpu\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ujwal\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`torch_dtype` is deprecated! Use `dtype` instead!


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Mean Reward
if telemetry_history["reward_mean"]:
    steps = list(range(1, len(telemetry_history["reward_mean"]) + 1))
    axes[0].plot(steps, telemetry_history["reward_mean"], marker="o", color="#2ecc71", linewidth=2)
    axes[0].set_title("Mean Reward per Logging Step", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Logged Step", fontsize=10)
    axes[0].set_ylabel("Total Reward (Format + Accuracy + Length)", fontsize=10)
    axes[0].grid(True, linestyle="--", alpha=0.6)

# Plot KL Divergence
if telemetry_history["kl"]:
    steps_kl = list(range(1, len(telemetry_history["kl"]) + 1))
    axes[1].plot(steps_kl, telemetry_history["kl"], marker="s", color="#e74c3c", linewidth=2)
    axes[1].set_title("Policy KL Divergence ($D_{KL}(\pi_\\theta \parallel \pi_{ref})$)", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Logged Step", fontsize=10)
    axes[1].set_ylabel("KL Value", fontsize=10)
    axes[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
from evaluate import evaluate_model

# Evaluate Trained GRPO Model on test split
results = evaluate_model(trainer.model, tokenizer, test_ds, num_samples_per_prompt=4)

print("\n" + "="*40)
print("       GRPO EVALUATION RESULTS")
print("="*40)
print(f"Pass@1 Accuracy:            {results['pass@1'] * 100:.2f}%")
print(f"Pass@4 (Majority) Accuracy: {results['pass@4'] * 100:.2f}%")
print(f"Average Reasoning Length:   {results['avg_reasoning_words']:.1f} words")
print("="*40)